In [7]:
import sys, glob, os
for d in glob.glob("/kaggle/input/*"):
    if os.path.isdir(os.path.join(d, "imgpref")):
        sys.path.insert(0, d); print("code at:", d); break

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| GPUs", torch.cuda.device_count())

!pip install -q -U timm

print("=== INPUT datasets ===")
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')

torch 2.10.0+cu128 | cuda True | GPUs 2
=== INPUT datasets ===
input/
  competitions/
    teta-nn-2-2026/
      sample_submission.csv
      train.parquet
      test.parquet
  datasets/
    elenaorehova/
      imgpref-v1-1/
        ARCHITECTURE.md
        README.md
        requirements.txt
        imgpref/
          data.py
          pair_features.py
          config.py
          utils.py
          features.py
          pipeline.py
          __init__.py
          submit.py
          cv.py
          models/
            gbdt.py
            finetune.py
            __init__.py
            bt_head.py
        configs/
          default.yaml
        notebooks/
          kaggle_run.py
      imgpref-feats/
        feat_test_image_1_vit_large_patch14_dinov2.npy
        feat_test_image_2_convnextv2_base.npy
        feat_test_image_1_vit_large_patch14_clip_224.npy
        feat_train_image_1_convnextv2_base.npy
        feat_train_image_2_vit_large_patch14_clip_224.npy
        feat_test_image_2_vit_l

In [13]:
from itertools import combinations
from imgpref.models.bt_head import train_bt_cv
from imgpref.models.gbdt import train_lgbm_cv
from imgpref.cv import search_blend_weights
from imgpref.utils import roc_auc, seed_everything
import torch

feats = {}
for bb in BACKBONES:
    feats[bb] = tuple(np.load(find(f"feat_{s}_{c}_{bb}.npy"))
                      for s in ["train","test"] for c in ["image_1","image_2"])
    feats[bb] = (feats[bb][0], feats[bb][1], feats[bb][2], feats[bb][3])

bt_res = {}
for bb, (a,b,c,d) in feats.items():
    seed_everything(42); torch.manual_seed(42)
    oof, test, _ = train_bt_cv(a, b, y, c, d, folds, cfg, "cpu")
    bt_res[bb] = (oof, test)
    print(f">>> {bb}: BT OOF = {roc_auc(y, oof):.5f}")

names = list(bt_res)
oofs  = [bt_res[n][0] for n in names]
w, auc = search_blend_weights(oofs, y)
print("PRED-BLEND (3 BT-головы):", round(auc,5), "| weights:", dict(zip(names, w)))

  [BT] fold 0: AUC=0.63653
  [BT] fold 1: AUC=0.64531
  [BT] fold 2: AUC=0.63689
  [BT] fold 3: AUC=0.65661
  [BT] fold 4: AUC=0.65213
  [BT] OOF AUC=0.64534
>>> convnextv2_base: BT OOF = 0.64534
  [BT] fold 0: AUC=0.63019
  [BT] fold 1: AUC=0.63145
  [BT] fold 2: AUC=0.63562
  [BT] fold 3: AUC=0.64461
  [BT] fold 4: AUC=0.65104
  [BT] OOF AUC=0.63786
>>> vit_large_patch14_dinov2: BT OOF = 0.63786
  [BT] fold 0: AUC=0.65433
  [BT] fold 1: AUC=0.66199
  [BT] fold 2: AUC=0.65134
  [BT] fold 3: AUC=0.65306
  [BT] fold 4: AUC=0.66333
  [BT] OOF AUC=0.65538
>>> vit_large_patch14_clip_224: BT OOF = 0.65538
PRED-BLEND (3 BT-головы): 0.66416 | weights: {'convnextv2_base': 0.3, 'vit_large_patch14_dinov2': 0.2, 'vit_large_patch14_clip_224': 0.5}


In [8]:
import glob, os, sys

cands = glob.glob("/kaggle/input/**/imgpref/__init__.py", recursive=True)
code_dir = os.path.dirname(os.path.dirname(cands[0]))
if code_dir not in sys.path:
    sys.path.insert(0, code_dir)

train = glob.glob("/kaggle/input/**/train.parquet", recursive=True)[0]
test  = glob.glob("/kaggle/input/**/test.parquet",  recursive=True)[0]
samp  = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)[0]
print("train:", train); print("test :", test); print("samp :", samp)

from imgpref.config import auto_config
cfg = auto_config(
    backbones=[{"name": "convnextv2_base.fcmae_ft_in22k_in1k_384", "img_size": 384, "batch_size": 48}],
    train_parquet=train,
    test_parquet=test,
    sample_submission=samp,
    work_dir="/kaggle/working/work",
    cache_dir="/kaggle/working/work/cache",
)
print("OK -> work:", cfg.work_dir)

train: /kaggle/input/competitions/teta-nn-2-2026/train.parquet
test : /kaggle/input/competitions/teta-nn-2-2026/test.parquet
samp : /kaggle/input/competitions/teta-nn-2-2026/sample_submission.csv
OK -> work: /kaggle/working/work


In [9]:
# import os, io, gc, ctypes, numpy as np, pandas as pd, torch
# import torch.nn.functional as F
# import pyarrow.parquet as pq
# from PIL import Image
# from concurrent.futures import ThreadPoolExecutor
# from tqdm.auto import tqdm
# from imgpref.features import build_model_and_transform, cache_path

# COL = cfg.img2_col     
# SAVE_CSV = True         
# NTHREADS = 8
# EXPORT = "/kaggle/working/export"; os.makedirs(EXPORT, exist_ok=True)
# try: libc = ctypes.CDLL("libc.so.6")
# except Exception: libc = None

# spec = cfg.backbones[0]
# bb = spec.name.split(".")[0]
# model, transform = build_model_and_transform(spec, cfg.pretrained, "cuda")
# if torch.cuda.device_count() > 1: model = torch.nn.DataParallel(model)
# model.eval()

# def decode_tf(b): return transform(Image.open(io.BytesIO(b)).convert("RGB"))
# def run_batch(ts):
#     x = torch.stack(ts).to("cuda", non_blocking=True)
#     with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.amp):
#         e = model(x).float()
#         if cfg.use_tta_hflip: e = (e + model(torch.flip(x, dims=[3])).float()) / 2
#     return F.normalize(e, dim=1).cpu().numpy().astype("float32")

# def extract(parquet, split):
#     out = cache_path(cfg, spec.name, split, COL)
#     npy_path = f"{EXPORT}/feat_{split}_{COL}_{bb}.npy"
#     csv_path = f"{EXPORT}/feat_{split}_{COL}_{bb}.csv"
#     if os.path.exists(npy_path):
#         if not os.path.exists(out): np.save(out, np.load(npy_path))  
#         print("exists:", npy_path); return
#     pf = pq.ParquetFile(parquet); feats, carry = [], []
#     pbar = tqdm(total=pf.metadata.num_rows, desc=f"{split}/{COL}")
#     with ThreadPoolExecutor(max_workers=NTHREADS) as ex:
#         for rb in pf.iter_batches(batch_size=128, columns=[COL]):
#             carry.extend(ex.map(decode_tf, rb.to_pydict()[COL])); del rb
#             while len(carry) >= spec.batch_size:
#                 feats.append(run_batch(carry[:spec.batch_size])); carry = carry[spec.batch_size:]
#                 pbar.update(spec.batch_size)
#                 if libc and len(feats) % 20 == 0: libc.malloc_trim(0)
#         if carry: feats.append(run_batch(carry)); pbar.update(len(carry))
#     pbar.close()
#     arr = np.concatenate(feats)
#     np.save(out, arr)            # кеш для run_train
#     np.save(npy_path, arr)       # скачиваемый .npy
#     if SAVE_CSV:
#         pd.DataFrame(arr, columns=[f"f{i}" for i in range(arr.shape[1])]).to_csv(csv_path, index=False)
#     print("saved:", npy_path, ("+csv " if SAVE_CSV else ""), arr.shape)
#     del feats, carry, arr, pf; gc.collect()
#     if libc: libc.malloc_trim(0)

# extract(cfg.train_parquet, "train")
# extract(cfg.test_parquet,  "test")
# print("COLUMN DONE:", COL)

In [10]:
# import os, io, gc, ctypes, numpy as np, torch
# import torch.nn.functional as F
# import pyarrow.parquet as pq
# from PIL import Image
# from concurrent.futures import ThreadPoolExecutor
# from tqdm.auto import tqdm
# from imgpref.config import BackboneSpec
# from imgpref.features import build_model_and_transform

# # BACKBONE = {"name": "vit_large_patch14_dinov2.lvd142m", "img_size": 224, "batch_size": 32}   
# BACKBONE = {"name": "vit_large_patch14_clip_224.openai", "img_size": 224, "batch_size": 48}  
# COL = cfg.img2_col      

# NTHREADS = 8
# EXPORT = "/kaggle/working/export"; os.makedirs(EXPORT, exist_ok=True)
# try: libc = ctypes.CDLL("libc.so.6")
# except Exception: libc = None

# spec = BackboneSpec(**BACKBONE)
# bb = spec.name.split(".")[0]
# model, transform = build_model_and_transform(spec, True, "cuda")
# if torch.cuda.device_count() > 1: model = torch.nn.DataParallel(model)
# model.eval()

# def decode_tf(b): return transform(Image.open(io.BytesIO(b)).convert("RGB"))
# def run_batch(ts):
#     x = torch.stack(ts).to("cuda", non_blocking=True)
#     with torch.no_grad(), torch.cuda.amp.autocast(enabled=True):
#         e = model(x).float()
#         e = (e + model(torch.flip(x, dims=[3])).float()) / 2      # hflip TTA
#     return F.normalize(e, dim=1).cpu().numpy().astype("float32")

# def extract(parquet, split):
#     npy_path = f"{EXPORT}/feat_{split}_{COL}_{bb}.npy"
#     if os.path.exists(npy_path): print("exists:", npy_path); return
#     pf = pq.ParquetFile(parquet); feats, carry = [], []
#     pbar = tqdm(total=pf.metadata.num_rows, desc=f"{bb} {split}/{COL}")
#     with ThreadPoolExecutor(max_workers=NTHREADS) as ex:
#         for rb in pf.iter_batches(batch_size=128, columns=[COL]):
#             carry.extend(ex.map(decode_tf, rb.to_pydict()[COL])); del rb
#             while len(carry) >= spec.batch_size:
#                 feats.append(run_batch(carry[:spec.batch_size])); carry = carry[spec.batch_size:]
#                 pbar.update(spec.batch_size)
#                 if libc and len(feats) % 20 == 0: libc.malloc_trim(0)
#         if carry: feats.append(run_batch(carry)); pbar.update(len(carry))
#     pbar.close()
#     arr = np.concatenate(feats); np.save(npy_path, arr); print("saved:", npy_path, arr.shape)
#     del feats, carry, arr, pf; gc.collect()
#     if libc: libc.malloc_trim(0)

# extract(cfg.train_parquet, "train")
# extract(cfg.test_parquet,  "test")
# print("DONE:", bb, COL)

In [11]:
import sys, glob, os, numpy as np
import pyarrow.parquet as pq

code_dir = next(os.path.dirname(os.path.dirname(p))
                for p in glob.glob("/kaggle/input/**/imgpref/__init__.py", recursive=True))
sys.path.insert(0, code_dir)
def find(name): return glob.glob(f"/kaggle/input/**/{name}", recursive=True)[0]

BACKBONES = ["convnextv2_base", "vit_large_patch14_dinov2", "vit_large_patch14_clip_224"]

def load_feats(split, col):
    return np.concatenate(
        [np.load(find(f"feat_{split}_{col}_{bb}.npy")) for bb in BACKBONES], axis=1)

F1_tr = load_feats("train", "image_1"); F2_tr = load_feats("train", "image_2")
F1_te = load_feats("test",  "image_1"); F2_te = load_feats("test",  "image_2")
y = np.asarray(pq.read_table(find("train.parquet"), columns=["is_image1_better"])
               .to_pydict()["is_image1_better"])
print("train:", F1_tr.shape, "test:", F1_te.shape, "| y mean:", y.mean().round(3))

train: (8710, 3072) test: (4290, 3072) | y mean: 0.347


In [14]:
import io, torch
from PIL import Image
from imgpref.pair_features import build_pair_features

def col_meta(parquet):
    pf = pq.ParquetFile(parquet); len1, len2, fmt1 = [], [], []
    for rb in pf.iter_batches(batch_size=256, columns=["image_1", "image_2"]):
        d = rb.to_pydict()
        for b1, b2 in zip(d["image_1"], d["image_2"]):
            len1.append(len(b1)); len2.append(len(b2))
            fmt1.append(Image.open(io.BytesIO(b1)).format or "UNK")
        del rb, d
    return {"len1": np.array(len1, np.float32), "len2": np.array(len2, np.float32),
            "fmt1": np.array(fmt1)}

meta_tr = col_meta(find("train.parquet"))
meta_te = col_meta(find("test.parquet"))

bt_oof, bt_test = {}, {}
for bb, (a, b, c, d) in feats.items():
    oo, tt = [], []
    for s in [42, 7, 2026]:
        seed_everything(s); torch.manual_seed(s)
        o, t, _ = train_bt_cv(a, b, y, c, d, folds, cfg, "cpu")
        oo.append(o); tt.append(t)
    bt_oof[bb] = np.mean(oo, axis=0); bt_test[bb] = np.mean(tt, axis=0)
    print(f">>> {bb} BT(3 seeds) OOF = {roc_auc(y, bt_oof[bb]):.5f}")

clip = "vit_large_patch14_clip_224"
Xtr1 = build_pair_features(feats[clip][0], feats[clip][1], meta_tr, mode="compact")
Xte1 = build_pair_features(feats[clip][2], feats[clip][3], meta_te, mode="compact")
g1_oof, g1_test, _ = train_lgbm_cv(Xtr1, y, Xte1, folds, cfg)

Xtr2 = build_pair_features(F1_tr, F2_tr, meta_tr, mode="compact")   
Xte2 = build_pair_features(F1_te, F2_te, meta_te, mode="compact")
g2_oof, g2_test, _ = train_lgbm_cv(Xtr2, y, Xte2, folds, cfg)

names = list(bt_oof) + ["gbdt_clip_meta", "gbdt_concat_meta"]
oofs  = list(bt_oof.values()) + [g1_oof, g2_oof]
tests = list(bt_test.values()) + [g1_test, g2_test]
w, auc = search_blend_weights(oofs, y)
print("FINAL BLEND OOF:", round(auc, 5))
print("weights:", dict(zip(names, np.round(w, 2))))

ranks = [np.argsort(np.argsort(t)).astype(float)/(len(t)-1) for t in tests]
write_submission(sum(wi*ri for wi, ri in zip(w, ranks)),
                 find("sample_submission.csv"), "/kaggle/working/submission.csv")

  [BT] fold 0: AUC=0.63653
  [BT] fold 1: AUC=0.64531
  [BT] fold 2: AUC=0.63689
  [BT] fold 3: AUC=0.65661
  [BT] fold 4: AUC=0.65213
  [BT] OOF AUC=0.64534
  [BT] fold 0: AUC=0.63157
  [BT] fold 1: AUC=0.64856
  [BT] fold 2: AUC=0.64411
  [BT] fold 3: AUC=0.64696
  [BT] fold 4: AUC=0.65543
  [BT] OOF AUC=0.64429
  [BT] fold 0: AUC=0.64224
  [BT] fold 1: AUC=0.65663
  [BT] fold 2: AUC=0.63344
  [BT] fold 3: AUC=0.65034
  [BT] fold 4: AUC=0.64927
  [BT] OOF AUC=0.64567
>>> convnextv2_base BT(3 seeds) OOF = 0.65064
  [BT] fold 0: AUC=0.63019
  [BT] fold 1: AUC=0.63145
  [BT] fold 2: AUC=0.63562
  [BT] fold 3: AUC=0.64461
  [BT] fold 4: AUC=0.65104
  [BT] OOF AUC=0.63786
  [BT] fold 0: AUC=0.62825
  [BT] fold 1: AUC=0.63170
  [BT] fold 2: AUC=0.64034
  [BT] fold 3: AUC=0.64702
  [BT] fold 4: AUC=0.65861
  [BT] OOF AUC=0.63981
  [BT] fold 0: AUC=0.62393
  [BT] fold 1: AUC=0.62265
  [BT] fold 2: AUC=0.63299
  [BT] fold 3: AUC=0.63895
  [BT] fold 4: AUC=0.65235
  [BT] OOF AUC=0.63274
>>> vi

'/kaggle/working/submission.csv'